In [ ]:
# ══════════════════════════════════════════════════════════════════
# Colab setup — self-contained. No git clone, no FORK_URL, no sys.path.
# Before running: Runtime → Change runtime type → T4 GPU (free tier)
# ══════════════════════════════════════════════════════════════════
!pip install -q kailash polars plotly gdown python-dotenv nest-asyncio httpx kailash-ml

import nest_asyncio; nest_asyncio.apply()

import torch
if torch.cuda.is_available():
    print(f'✓ GPU available: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU — training will be slow. Runtime → Change runtime type → T4 GPU')

print('✓ Setup complete. All helpers defined in the next cell.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# MLFP inlined helpers — DO NOT EDIT (collapse this cell!)
# Auto-generated by scripts/generate_selfcontained_notebook.py for mlfp05
# ══════════════════════════════════════════════════════════════════
from __future__ import annotations

# ── shared/kailash_helpers.py ──
"""Common Kailash SDK setup patterns for MLFP exercises."""


import logging
import os
from dataclasses import dataclass, field, replace
from pathlib import Path
from typing import Any, Literal

import numpy as np
import polars as pl
from dotenv import load_dotenv

logger = logging.getLogger(__name__)


def setup_environment() -> None:
    """Load .env and validate common configuration.

    Call this at the top of every exercise that needs API keys or DB connections.
    """
    # Find .env by walking up from the exercise file
    env_path = Path.cwd() / ".env"
    if not env_path.exists():
        # Try repo root
        for parent in Path.cwd().parents:
            candidate = parent / ".env"
            if candidate.exists():
                env_path = candidate
                break

    load_dotenv(env_path)


def get_connection_manager(db_url: str | None = None):
    """Create a ConnectionManager for kailash-ml engines.

    Args:
        db_url: Database URL. Defaults to SQLite at ./mlfp.db
    """
    from kailash.db import ConnectionManager

    url = db_url or os.environ.get("DATABASE_URL", "sqlite:///mlfp.db")
    return ConnectionManager(url)


def get_device() -> "torch.device":
    """Return the best available compute device as a ``torch.device``.

    Routes through ``kailash_ml.device()`` (the canonical kailash-ml 0.12+
    accelerator detector) so MPS / CUDA / ROCm / Intel XPU / CPU are all
    selected by the same logic the rest of the platform uses. The previous
    hand-rolled ``mps→cuda→cpu`` cascade is replaced because:

      * kailash-ml's detector knows about ROCm, Intel XPU, and fp16/bf16
        capability flags — the cascade in this helper did not.
      * Apple-Silicon students get the Metal Performance Shaders backend
        with mixed-precision (fp16) without any opt-in.
      * One detection point means lessons that print "Using device: …"
        agree with what kailash-ml's MLEngine() actually picks.
    """
    import kailash_ml as km
    import torch

    backend = km.device()  # BackendInfo (auto MPS on Mac, CUDA on Linux+NVIDIA, …)
    return torch.device(backend.device_string)


def get_llm_model() -> str:
    """Get the configured LLM model name from environment."""
    setup_environment()
    model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
    if not model:
        raise EnvironmentError(
            "No LLM model configured. Set DEFAULT_LLM_MODEL or OPENAI_PROD_MODEL in .env"
        )
    return model


# ════════════════════════════════════════════════════════════════════════
# SPLIT FIRST, THEN PREPROCESS
# ════════════════════════════════════════════════════════════════════════
# kailash-ml's ``PreprocessingPipeline.setup()`` fits its imputer, its
# categorical encoder (including TARGET encoding) and its scaler on EVERY
# row it is given, and only afterwards splits train/test. Handing it the
# whole dataset therefore lets test-set statistics leak into training.
#
# The helpers below hold the test rows out FIRST, fit the pipeline on the
# training rows only, and push the held-out rows through the fitted
# ``transform()``. ``setup()`` still splits internally, but that split now
# happens inside the training rows, so it cannot leak anything.

_SETUP_OPTIONS: frozenset[str] = frozenset(
    {
        "normalize",
        "normalize_method",
        "categorical_encoding",
        "imputation_strategy",
        "impute_n_neighbors",
        "remove_outliers",
        "outlier_threshold",
        "max_cardinality",
        "exclude_columns",
        "remove_multicollinearity",
        "multicollinearity_threshold",
        "pca",
        "pca_components",
        "transform_target",
        "fix_imbalance",
        "imbalance_method",
    }
)

# Arguments the helpers own themselves; passing them through would
# re-introduce the leak (``data`` = the full frame) or fight the split.
_OWNED_BY_HELPER: frozenset[str] = frozenset({"data", "target", "train_size", "seed"})


@dataclass(frozen=True)
class PreprocessResult:
    """Outcome of :func:`split_then_preprocess` / :func:`preprocess_train_test`.

    Mirrors the fields of kailash-ml's ``SetupResult`` that exercises use,
    plus the fitted pipeline and a record of what was done to which split.

    ``train_data`` / ``test_data``
        Both produced by the SAME pipeline, fitted on the training rows only.
    ``transformers``
        The fitted imputer / encoder / scaler state (e.g.
        ``transformers["ordinal_mappings"]``), learned from training rows.
    ``train_only_steps``
        Steps applied to the training split only (outlier removal,
        class-imbalance resampling). The test split is never filtered or
        resampled — it stays a faithful sample of the data.
    ``test_null_fills``
        Columns that were complete in the training rows but have gaps in
        the test rows (the fitted pipeline has no learned fill for them),
        mapped to the TRAINING statistic used to fill them.
    """

    train_data: pl.DataFrame
    test_data: pl.DataFrame
    target_column: str
    task_type: str
    numeric_columns: list[str]
    categorical_columns: list[str]
    transformers: dict[str, Any]
    pipeline: Any
    original_shape: tuple[int, int]
    train_raw_rows: int
    test_raw_rows: int
    stratified: bool
    train_only_steps: tuple[str, ...] = ()
    test_null_fills: dict[str, Any] = field(default_factory=dict)
    summary: str = ""


def _looks_like_classification(series: pl.Series) -> bool:
    """Same rule kailash-ml uses to call a target "classification"."""
    if series.dtype in (pl.Boolean, pl.Categorical, pl.Utf8, pl.String):
        return True
    return series.drop_nulls().n_unique() <= 20


def split_raw_train_test(
    df: pl.DataFrame,
    target: str,
    *,
    test_size: float | int = 0.2,
    seed: int = 42,
    stratify: bool | Literal["auto"] = "auto",
) -> tuple[pl.DataFrame, pl.DataFrame, bool]:
    """Hold out test rows from the RAW frame before anything is fitted.

    Rows are shuffled with ``seed``. ``test_size`` is a fraction in (0, 1)
    or an exact row count. With ``stratify="auto"`` a classification target
    (same rule as kailash-ml) is stratified so every class keeps its share
    in both splits; ``True`` / ``False`` force the choice.

    Returns ``(train_raw, test_raw, stratified)``; both splits keep the
    shuffled order.
    """
    if target not in df.columns:
        raise ValueError(f"Target column '{target}' not found in data.")
    n = df.height
    if n < 2:
        raise ValueError(f"Need at least 2 rows to split, got {n}.")
    if isinstance(test_size, np.integer):
        test_size = int(test_size)
    if isinstance(test_size, bool) or not isinstance(test_size, (int, float)):
        raise TypeError(
            f"test_size must be a fraction in (0, 1) or a row count, got {test_size!r}."
        )
    if isinstance(test_size, int):
        if not 1 <= test_size < n:
            raise ValueError(f"test_size as a row count must be in [1, {n - 1}], got {test_size}.")
        test_fraction = test_size / n
    else:
        if not 0.0 < test_size < 1.0:
            raise ValueError(f"test_size as a fraction must be in (0, 1), got {test_size}.")
        test_fraction = float(test_size)

    is_classification = _looks_like_classification(df[target])
    if stratify == "auto":
        stratified = is_classification
    elif stratify is True:
        if not is_classification:
            raise ValueError(
                f"stratify=True needs a classification target; '{target}' has "
                f"{df[target].drop_nulls().n_unique()} distinct values."
            )
        stratified = True
    elif stratify is False:
        stratified = False
    else:
        raise TypeError(f"stratify must be True, False or 'auto', got {stratify!r}.")

    perm = np.random.RandomState(seed).permutation(n)
    shuffled = df[perm.tolist()]

    if stratified:
        # Within each class (in shuffled order), the first round(n_c * f)
        # rows go to test — the class share is the same in both splits.
        is_test = pl.int_range(pl.len()).over(target) < (
            pl.len().over(target) * test_fraction
        ).round()
        flagged = shuffled.with_columns(is_test.alias("__split_is_test"))
        test_raw = flagged.filter(pl.col("__split_is_test")).drop("__split_is_test")
        train_raw = flagged.filter(~pl.col("__split_is_test")).drop("__split_is_test")
    else:
        n_test = test_size if isinstance(test_size, int) else int(round(n * test_fraction))
        train_raw = shuffled.head(n - n_test)
        test_raw = shuffled.tail(n_test)

    if train_raw.height == 0 or test_raw.height == 0:
        raise ValueError(
            f"Split produced an empty side (train={train_raw.height}, "
            f"test={test_raw.height}); adjust test_size."
        )
    return train_raw, test_raw, stratified


def _fill_unfitted_test_nulls(
    test_raw: pl.DataFrame,
    train_raw: pl.DataFrame,
    fitted: Any,
    strategy: str,
) -> tuple[pl.DataFrame, dict[str, Any]]:
    """Fill test gaps the fitted pipeline has no learned value for.

    kailash-ml only stores a fill value for columns that had gaps in the
    rows it was fitted on. A column that is complete in the training rows
    but has gaps in the test rows would otherwise reach the model as null.
    The fill is the TRAINING statistic for the same strategy (never a test
    statistic), and every such column is logged and returned.
    """
    if strategy == "drop":
        return test_raw, {}  # transform() drops incomplete rows itself
    learned = set(fitted.transformers.get("imputer_stats", {}))
    if strategy in ("knn", "iterative"):
        learned |= set(fitted.transformers.get("sklearn_imputer_cols", []))

    fills: dict[str, Any] = {}
    for col in fitted.numeric_columns + fitted.categorical_columns:
        if col in learned or col not in test_raw.columns:
            continue
        if test_raw[col].null_count() == 0:
            continue
        train_col = train_raw[col].drop_nulls()
        if col in fitted.categorical_columns or strategy == "mode":
            modes = train_col.mode().sort()
            if len(modes) == 0:
                raise ValueError(f"Cannot fill '{col}': no observed training values.")
            value = modes[0]
        elif strategy == "median":
            value = train_col.median()
        else:  # mean (and the numeric fallback for any other strategy)
            value = train_col.mean()
        if value is None:
            raise ValueError(f"Cannot fill '{col}': no observed training values.")
        fills[col] = value
    if fills:
        logger.warning(
            "Test rows have gaps in columns that were complete in training; "
            "filled with TRAINING statistics: %s",
            fills,
        )
        test_raw = test_raw.with_columns(
            [pl.col(c).fill_null(v) for c, v in fills.items()]
        )
    return test_raw, fills


def preprocess_train_test(
    train_raw: pl.DataFrame,
    test_raw: pl.DataFrame,
    target: str,
    *,
    seed: int = 42,
    **setup_kwargs: Any,
) -> PreprocessResult:
    """Fit ``PreprocessingPipeline`` on ``train_raw`` only; transform both.

    ``setup_kwargs`` are ``PreprocessingPipeline.setup()`` options
    (``normalize``, ``categorical_encoding``, ``imputation_strategy`` …).
    Steps that change the ROWS of a split are applied to training rows only:

    * ``remove_outliers`` — outlier rows are dropped from training only.
    * ``fix_imbalance`` — SMOTE / ADASYN (or the class-weight flag) is
      applied to the whole transformed training split, never to test.

    Use this directly inside cross-validation folds; use
    :func:`split_then_preprocess` for a single train/test split.
    """
    from kailash_ml import PreprocessingPipeline

    owned = sorted(_OWNED_BY_HELPER & set(setup_kwargs))
    if owned:
        raise TypeError(
            f"{owned} are set by the split-first helper itself and cannot be "
            "passed through to setup()."
        )
    unknown = sorted(set(setup_kwargs) - _SETUP_OPTIONS)
    if unknown:
        raise TypeError(f"Unknown PreprocessingPipeline.setup() options: {unknown}.")
    for name, frame in (("train_raw", train_raw), ("test_raw", test_raw)):
        if target not in frame.columns:
            raise ValueError(f"Target column '{target}' not found in {name}.")
    if train_raw.height == 0 or test_raw.height == 0:
        raise ValueError("train_raw and test_raw must both have rows.")
    if set(train_raw.columns) != set(test_raw.columns):
        raise ValueError("train_raw and test_raw must have the same columns.")

    fix_imbalance = bool(setup_kwargs.pop("fix_imbalance", False))
    imbalance_method = setup_kwargs.get("imbalance_method", "smote")
    remove_outliers = bool(setup_kwargs.get("remove_outliers", False))

    pipeline = PreprocessingPipeline()
    # The pipeline only ever sees training rows. Its internal split stays
    # inside them; resampling is applied below to the full training split.
    fitted = pipeline.setup(
        data=train_raw, target=target, seed=seed, fix_imbalance=False, **setup_kwargs
    )

    train_only: list[str] = []
    if remove_outliers:
        # setup() removed the outlier rows (training rows only); its two
        # internal pieces together are every surviving training row.
        train_data = pl.concat([fitted.train_data, fitted.test_data], how="vertical")
        train_only.append(
            f"remove_outliers: {train_raw.height - train_data.height} training rows dropped"
        )
    else:
        train_data = pipeline.transform(train_raw)

    strategy = setup_kwargs.get("imputation_strategy", "mean")
    test_filled, test_fills = _fill_unfitted_test_nulls(test_raw, train_raw, fitted, strategy)
    test_data = pipeline.transform(test_filled)

    if fix_imbalance and fitted.task_type == "classification":
        resample = getattr(pipeline, "_apply_imbalance_correction", None)
        if resample is None:
            raise RuntimeError(
                "This kailash-ml version has no imbalance-correction step on "
                "PreprocessingPipeline; cannot apply fix_imbalance to the training split."
            )
        before = train_data.height
        train_data = resample(train_data, target, imbalance_method, seed)
        train_only.append(
            f"fix_imbalance ({imbalance_method}): training rows {before} -> {train_data.height}"
        )

    summary = "\n".join(
        [
            f"Task type: {fitted.task_type}",
            f"Split first: {train_raw.height} train rows / {test_raw.height} test rows "
            "(pipeline fitted on the train rows only)",
            f"Numeric features: {len(fitted.numeric_columns)}",
            f"Categorical features: {len(fitted.categorical_columns)}",
            f"Encoding: {setup_kwargs.get('categorical_encoding', 'onehot')}",
            f"Normalization: {'yes' if setup_kwargs.get('normalize', True) else 'no'}",
            f"Imputation: {strategy}",
            f"Train-only steps: {'; '.join(train_only) if train_only else 'none'}",
            f"Transformed train: {train_data.height} rows x {train_data.width} cols",
            f"Transformed test: {test_data.height} rows x {test_data.width} cols",
        ]
    )

    return PreprocessResult(
        train_data=train_data,
        test_data=test_data,
        target_column=target,
        task_type=fitted.task_type,
        numeric_columns=list(fitted.numeric_columns),
        categorical_columns=list(fitted.categorical_columns),
        transformers=dict(fitted.transformers),
        pipeline=pipeline,
        original_shape=(train_raw.height + test_raw.height, train_raw.width),
        train_raw_rows=train_raw.height,
        test_raw_rows=test_raw.height,
        stratified=False,
        train_only_steps=tuple(train_only),
        test_null_fills=test_fills,
        summary=summary,
    )


def split_then_preprocess(
    df: pl.DataFrame,
    target: str,
    *,
    test_size: float | int = 0.2,
    seed: int = 42,
    stratify: bool | Literal["auto"] = "auto",
    **setup_kwargs: Any,
) -> PreprocessResult:
    """Leak-free replacement for ``PreprocessingPipeline().setup(df, ...)``.

    1. Split the RAW frame first (seeded shuffle; stratified on the target
       for classification when ``stratify="auto"``).
    2. Fit ``PreprocessingPipeline`` on the training rows only.
    3. ``transform()`` the test rows with the fitted pipeline.

    ``setup_kwargs`` are passed to ``PreprocessingPipeline.setup()``; see
    :func:`preprocess_train_test` for how row-changing options are handled.
    """
    train_raw, test_raw, stratified = split_raw_train_test(
        df, target, test_size=test_size, seed=seed, stratify=stratify
    )
    result = preprocess_train_test(train_raw, test_raw, target, seed=seed, **setup_kwargs)
    split_line = "stratified on the target" if stratified else "random"
    return replace(
        result,
        stratified=stratified,
        summary=result.summary.replace("Split first:", f"Split first ({split_line}):", 1),
    )


# ── shared/mlfp05/ex_5.py ──
"""
Shared utilities for Exercise 5 — GANs and Generative Models.

Infrastructure only: data loading, visualisation helpers, metric computation,
and kailash-ml engine setup. No domain logic or business scenarios.
"""

import asyncio
import pickle
from pathlib import Path
from typing import TYPE_CHECKING

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import torchvision
import matplotlib.pyplot as plt
from matplotlib.figure import Figure

from kailash.db import ConnectionManager
from kailash_ml import ExperimentTracker
from kailash_ml import ModelRegistry
from kailash_ml.types import MetricSpec


if TYPE_CHECKING:
    from kailash_ml.engines.model_registry import ModelVersion

# ════════════════════════════════════════════════════════════════════════
# Constants
# ════════════════════════════════════════════════════════════════════════
LATENT_DIM = 64
IMG_DIM = 28 * 28
BATCH_SIZE = 128
try:
    _HERE = Path(__file__).resolve()
    REPO_ROOT = _HERE.parents[2]
except NameError:
    REPO_ROOT = Path.cwd()
DATA_DIR = REPO_ROOT / "data" / "mlfp05" / "mnist"
OUTPUT_DIR = Path("outputs") / "ex5_gans"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# ════════════════════════════════════════════════════════════════════════
# Environment and device
# ════════════════════════════════════════════════════════════════════════
def init_environment() -> torch.device:
    """Set up environment, seeds, and return the compute device."""
    setup_environment()
    torch.manual_seed(42)
    np.random.seed(42)
    device = get_device()
    print(f"Using device: {device}")
    return device


# ════════════════════════════════════════════════════════════════════════
# Data loading
# ════════════════════════════════════════════════════════════════════════
def load_mnist(device: torch.device) -> tuple[torch.Tensor, torch.Tensor, DataLoader]:
    """Load full MNIST (60K), scale to [-1, 1] for tanh generators.

    Returns:
        X_real: (60000, 1, 28, 28) tensor on device, range [-1, 1]
        y_real: (60000,) long tensor on device
        real_loader: DataLoader with batch_size=128, shuffle, drop_last
    """
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    train_set = torchvision.datasets.MNIST(
        root=str(DATA_DIR),
        train=True,
        download=True,
        transform=torchvision.transforms.ToTensor(),
    )

    X_real = torch.stack([train_set[i][0] for i in range(len(train_set))])
    y_real = torch.tensor(
        [train_set[i][1] for i in range(len(train_set))], dtype=torch.long
    )
    X_real = (X_real * 2.0 - 1.0).to(device)
    y_real = y_real.to(device)

    print(
        f"MNIST: {len(X_real)} digits, shape {tuple(X_real.shape[1:])}, "
        f"pixel range [{X_real.min():.2f}, {X_real.max():.2f}]"
    )
    class_dist = ", ".join(f"{c}={int((y_real == c).sum())}" for c in range(10))
    print(f"  class distribution: {class_dist}")

    real_loader = DataLoader(
        TensorDataset(X_real), batch_size=BATCH_SIZE, shuffle=True, drop_last=True
    )
    return X_real, y_real, real_loader


def load_mnist_test(device: torch.device) -> tuple[torch.Tensor, torch.Tensor]:
    """Load the 10K MNIST test digits — images no generator was trained on.

    Returns:
        X_test: (10000, 1, 28, 28) tensor on device, range [-1, 1]
        y_test: (10000,) long tensor on device
    """
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    test_set = torchvision.datasets.MNIST(
        root=str(DATA_DIR),
        train=False,
        download=True,
        transform=torchvision.transforms.ToTensor(),
    )
    X_test = torch.stack([test_set[i][0] for i in range(len(test_set))])
    y_test = torch.tensor(
        [test_set[i][1] for i in range(len(test_set))], dtype=torch.long
    )
    return (X_test * 2.0 - 1.0).to(device), y_test.to(device)


# ════════════════════════════════════════════════════════════════════════
# Kailash engine setup
# ════════════════════════════════════════════════════════════════════════
def setup_engines() -> (
    tuple[ConnectionManager, ExperimentTracker, str, ModelRegistry | None]
):
    """Create ExperimentTracker (kailash-ml 1.1.1 factory) and ModelRegistry.

    Returns:
        conn, tracker, experiment_name, registry
    """

    async def _setup():
        # Schema-conflict workaround (kailash-ml 1.5.x): ExperimentTracker
        # and ModelRegistry use incompatible _kml_model_versions schemas.
        # Route them to separate sqlite files until upstream fixes the conflict.
        db = "sqlite:///mlfp05_gans.db"
        registry_db = "sqlite:///mlfp05_gans_registry.db"
        tracker = await ExperimentTracker.create(store_url=db)
        conn = ConnectionManager(registry_db)
        await conn.initialize()
        registry = ModelRegistry(conn)
        return conn, tracker, "m5_gans", registry

    return asyncio.run(_setup())


async def close_engines(conn: ConnectionManager) -> None:
    """Cleanly shut down the connection manager."""
    await conn.close()


# ════════════════════════════════════════════════════════════════════════
# Generator and Discriminator architectures
# ════════════════════════════════════════════════════════════════════════
class Generator(nn.Module):
    """MLP Generator: z -> 784-d -> (1, 28, 28).

    A fully-connected (MLP) generator, not a convolutional DCGAN. It borrows
    the DCGAN training recipe — BatchNorm + LeakyReLU in hidden layers and a
    Tanh output to match the [-1, 1] image scaling. Linear layers that feed
    BatchNorm have no bias: BatchNorm subtracts the batch mean, so such a
    bias would be cancelled and would receive exactly zero gradient (which
    the Prescription Pad would report as a vanishing-gradient layer).
    """

    def __init__(self, latent_dim: int = LATENT_DIM, hidden: int = 256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim, hidden, bias=False),
            nn.BatchNorm1d(hidden),
            nn.LeakyReLU(0.2),
            nn.Linear(hidden, hidden * 2, bias=False),
            nn.BatchNorm1d(hidden * 2),
            nn.LeakyReLU(0.2),
            nn.Linear(hidden * 2, IMG_DIM),
            nn.Tanh(),
        )

    def forward(self, z: torch.Tensor) -> torch.Tensor:
        return self.net(z).view(-1, 1, 28, 28)


class Discriminator(nn.Module):
    """MLP Discriminator: 28x28 -> scalar logit.

    Dropout prevents D from overfitting to real images (memorising
    instead of learning distributional features).
    """

    def __init__(self, hidden: int = 256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(IMG_DIM, hidden * 2),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
            nn.Linear(hidden * 2, hidden),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
            nn.Linear(hidden, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


# ════════════════════════════════════════════════════════════════════════
# FID feature extractor
# ════════════════════════════════════════════════════════════════════════
class LeNetFeatureExtractor(nn.Module):
    """CNN feature extractor for FID computation.

    Returns 64-dim feature vectors (analogous to InceptionV3 pool3 layer).
    We use a domain-specific extractor because InceptionV3 expects 299x299 RGB;
    for 28x28 grayscale MNIST a trained LeNet gives more meaningful distances.
    """

    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, 5, stride=2, padding=2),
            nn.ReLU(),
            nn.Conv2d(16, 32, 5, stride=2, padding=2),
            nn.ReLU(),
            nn.Flatten(),
            nn.Linear(32 * 7 * 7, 64),
            nn.ReLU(),
        )
        self.classifier = nn.Linear(64, 10)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.classifier(self.features(x))

    def extract_features(self, x: torch.Tensor) -> torch.Tensor:
        return self.features(x)


def train_feature_extractor(
    X_real: torch.Tensor,
    y_real: torch.Tensor,
    device: torch.device,
    epochs: int = 5,
) -> LeNetFeatureExtractor:
    """Train the LeNet feature extractor on MNIST for FID computation.

    Returns the trained extractor in eval mode.
    """
    print("\n== Training feature extractor (for FID + mode coverage) ==")
    extractor = LeNetFeatureExtractor().to(device)
    opt = torch.optim.Adam(extractor.parameters(), lr=1e-3)
    X_01 = (X_real + 1.0) / 2.0  # [0, 1] for classifier

    for epoch in range(epochs):
        losses = []
        for xb, yb in DataLoader(
            TensorDataset(X_01, y_real), batch_size=256, shuffle=True
        ):
            loss = F.cross_entropy(extractor(xb), yb)
            opt.zero_grad()
            loss.backward()
            opt.step()
            losses.append(loss.item())
        with torch.no_grad():
            acc = (extractor(X_01[:10000]).argmax(-1) == y_real[:10000]).float().mean()
        print(f"  epoch {epoch+1}/{epochs}  loss={np.mean(losses):.3f}  acc={acc:.3f}")

    extractor.eval()
    return extractor


# ════════════════════════════════════════════════════════════════════════
# FID computation
# ════════════════════════════════════════════════════════════════════════
def compute_fid(
    extractor: LeNetFeatureExtractor,
    real: torch.Tensor,
    generated: torch.Tensor,
) -> float:
    """Frechet distance between real and generated feature distributions.

    FID = ||mu_r - mu_g||^2 + Tr(Sigma_r + Sigma_g - 2*sqrt(Sigma_r @ Sigma_g))

    Computed in the 64-d LeNet feature space (not Inception), so values are
    only comparable with other values from the same extractor. Lower = the
    generated distribution is closer to the real one (fidelity AND
    diversity). Uses eigenvalues of Sigma_r @ Sigma_g (no scipy dependency).
    """
    extractor.eval()

    def _extract(images: torch.Tensor) -> np.ndarray:
        feats = []
        with torch.no_grad():
            for i in range(0, len(images), 512):
                feats.append(
                    extractor.extract_features(images[i : i + 512]).cpu().numpy()
                )
        return np.concatenate(feats)

    rf, gf = _extract(real), _extract(generated)
    mu_r, mu_g = rf.mean(0), gf.mean(0)
    sig_r, sig_g = np.cov(rf, rowvar=False), np.cov(gf, rowvar=False)

    diff = mu_r - mu_g
    # Sigma_r @ Sigma_g is a product of two symmetric PSD matrices — it is
    # NOT itself symmetric, so np.linalg.eigh (Hermitian-only) reads the wrong
    # triangle and returns garbage (this produced large-negative FID). Its
    # eigenvalues are nonetheless real and non-negative (it is similar to the
    # symmetric PSD matrix Sigma_r^½ Sigma_g Sigma_r^½), so use the general
    # eigensolver and the identity Tr(sqrt(AB)) = Σ sqrt(λ_i(AB)).
    eigvals = np.linalg.eigvals(sig_r @ sig_g)
    eigvals = np.maximum(np.real(eigvals), 0.0)
    trace_sqrt = float(np.sum(np.sqrt(eigvals)))

    return float(diff @ diff + np.trace(sig_r) + np.trace(sig_g) - 2 * trace_sqrt)


# ════════════════════════════════════════════════════════════════════════
# Mode coverage diagnostic
# ════════════════════════════════════════════════════════════════════════
def mode_coverage(
    G: Generator,
    classifier: LeNetFeatureExtractor,
    device: torch.device,
    n: int = 5000,
) -> tuple[int, dict[int, int], float]:
    """Measure mode coverage: how many of the 10 digit classes the generator produces.

    Returns:
        n_classes: number of unique classes generated
        per_class_counts: dict mapping class -> count
        shannon_entropy: diversity measure (max = log2(10) = 3.32 for uniform)
    """
    G.eval()
    classifier.eval()
    with torch.no_grad():
        fake_01 = (G(torch.randn(n, LATENT_DIM, device=device)) + 1) / 2
        preds = classifier(fake_01).argmax(-1).cpu().numpy()
    unique, counts = np.unique(preds, return_counts=True)
    probs = counts / counts.sum()
    entropy = float(-np.sum(probs * np.log2(probs + 1e-10)))
    return int(len(unique)), {int(k): int(v) for k, v in zip(unique, counts)}, entropy


# ════════════════════════════════════════════════════════════════════════
# Visualisation helpers
# ════════════════════════════════════════════════════════════════════════
def plot_image_grid(
    images: torch.Tensor,
    nrow: int = 8,
    ncol: int = 8,
    title: str = "Generated Images",
    save_path: str | None = None,
) -> Figure:
    """Plot an 8x8 grid of generated images.

    Args:
        images: (N, 1, 28, 28) tensor in [-1, 1] range
        nrow, ncol: grid dimensions
        title: figure title
        save_path: optional path to save the figure
    """
    n = min(nrow * ncol, len(images))
    fig, axes = plt.subplots(nrow, ncol, figsize=(12, 12))
    fig.suptitle(title, fontsize=16, fontweight="bold")

    for i in range(nrow * ncol):
        ax = axes[i // ncol][i % ncol]
        if i < n:
            img = images[i].detach().cpu().squeeze().numpy()
            img = (img + 1) / 2  # [-1,1] -> [0,1]
            ax.imshow(img, cmap="gray", vmin=0, vmax=1)
        ax.axis("off")

    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"  Saved: {save_path}")
    return fig


def plot_latent_interpolation(
    G: Generator,
    device: torch.device,
    n_steps: int = 10,
    n_rows: int = 5,
    title: str = "Latent Space Interpolation",
    save_path: str | None = None,
) -> Figure:
    """Interpolate between pairs of random latent vectors.

    Shows smooth transitions between generated images — evidence that
    the generator has learned a continuous manifold, not memorised digits.
    """
    G.eval()
    fig, axes = plt.subplots(n_rows, n_steps, figsize=(n_steps * 1.2, n_rows * 1.4))
    fig.suptitle(title, fontsize=14, fontweight="bold")

    with torch.no_grad():
        for row in range(n_rows):
            z1 = torch.randn(1, LATENT_DIM, device=device)
            z2 = torch.randn(1, LATENT_DIM, device=device)
            for col in range(n_steps):
                alpha = col / (n_steps - 1)
                z = (1 - alpha) * z1 + alpha * z2
                img = G(z).squeeze().cpu().numpy()
                img = (img + 1) / 2
                axes[row][col].imshow(img, cmap="gray", vmin=0, vmax=1)
                axes[row][col].axis("off")

    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"  Saved: {save_path}")
    return fig


def plot_training_progression(
    G: Generator,
    device: torch.device,
    epoch_snapshots: dict[int, dict],
    title: str = "Training Progression",
    save_path: str | None = None,
) -> Figure:
    """Plot generated images at different training epochs.

    Shows how generation quality improves over training — from random
    noise to recognisable digits.

    Args:
        epoch_snapshots: dict of {epoch: state_dict} captured during training
    """
    n_snapshots = len(epoch_snapshots)
    n_samples = 8
    fig, axes = plt.subplots(
        n_snapshots, n_samples, figsize=(n_samples * 1.4, n_snapshots * 1.6)
    )
    fig.suptitle(title, fontsize=14, fontweight="bold")

    fixed_z = torch.randn(n_samples, LATENT_DIM, device=device)

    for row, (epoch, state_dict) in enumerate(sorted(epoch_snapshots.items())):
        G.load_state_dict(state_dict)
        G.eval()
        with torch.no_grad():
            imgs = G(fixed_z)
        for col in range(n_samples):
            ax = axes[row][col] if n_snapshots > 1 else axes[col]
            img = imgs[col].squeeze().cpu().numpy()
            img = (img + 1) / 2
            ax.imshow(img, cmap="gray", vmin=0, vmax=1)
            ax.axis("off")
            if col == 0:
                ax.set_ylabel(f"Epoch {epoch}", fontsize=10, rotation=0, labelpad=50)

    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"  Saved: {save_path}")
    return fig


def plot_loss_curves(
    g_losses: list[float],
    d_losses: list[float],
    title: str = "Training Dynamics",
    g_label: str = "Generator",
    d_label: str = "Discriminator",
    save_path: str | None = None,
) -> Figure:
    """Plot G vs D loss curves across epochs."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(title, fontsize=14, fontweight="bold")

    epochs = range(1, len(g_losses) + 1)

    # Individual losses
    ax1.plot(epochs, g_losses, "b-", linewidth=2, label=g_label)
    ax1.plot(epochs, d_losses, "r-", linewidth=2, label=d_label)
    ax1.set_xlabel("Epoch", fontsize=12)
    ax1.set_ylabel("Loss", fontsize=12)
    ax1.set_title("G vs D Loss", fontsize=13)
    ax1.legend(fontsize=11)
    ax1.grid(True, alpha=0.3)

    # G/D ratio — healthy GAN has ratio near 1
    ratio = [g / (d + 1e-8) for g, d in zip(g_losses, d_losses)]
    ax2.plot(epochs, ratio, "g-", linewidth=2)
    ax2.axhline(y=1.0, color="gray", linestyle="--", alpha=0.5, label="Balanced (1.0)")
    ax2.set_xlabel("Epoch", fontsize=12)
    ax2.set_ylabel("G/D Loss Ratio", fontsize=12)
    ax2.set_title("Training Balance", fontsize=13)
    ax2.legend(fontsize=11)
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"  Saved: {save_path}")
    return fig


# ════════════════════════════════════════════════════════════════════════
# Model registration helper
# ════════════════════════════════════════════════════════════════════════
def register_generator(
    registry: ModelRegistry | None,
    name: str,
    model: Generator,
    fid: float,
    coverage: int,
    entropy: float,
) -> "ModelVersion | None":
    """Register a trained generator in ModelRegistry with quality metrics."""
    if registry is None:
        print(f"  ModelRegistry not available — skipping {name}")
        return None

    async def _register():
        ver = await registry.register_model(
            name=f"m5_{name}",
            artifact=pickle.dumps(model.state_dict()),
            metrics=[
                MetricSpec(name="fid_score", value=fid),
                MetricSpec(name="mode_coverage", value=float(coverage)),
                MetricSpec(name="class_entropy", value=entropy),
            ],
        )
        print(
            f"  Registered {name}: v={ver.version}, FID={fid:.2f}, "
            f"coverage={coverage}/10"
        )
        return ver

    return asyncio.run(_register())




# ════════════════════════════════════════════════════════════════════════
# MLFP05 Exercise 5.4 — DCGAN: Convolutional GAN for MNIST
# ════════════════════════════════════════════════════════════════════════
#
# WHAT YOU'LL LEARN:
#   After completing this file, you will be able to:
#   - Explain WHY convolutional GANs dominate image generation: weight
#     sharing matches the spatial structure of images (a "3" is a "3"
#     wherever it sits in the frame)
#   - Build the DCGAN pair (Radford et al. 2016): ConvTranspose2d
#     generator with BatchNorm+ReLU+Tanh; strided-conv discriminator
#     with LeakyReLU — no pooling layers anywhere
#   - Apply the DCGAN training recipe: Adam(2e-4, betas=(0.5, 0.999)),
#     BatchNorm in both nets, no bias into BatchNorm layers
#   - Measure quality honestly with FID (LeNet feature space) and mode
#     coverage — never "the grid looks nice" alone
#   - Compare against the MLP GAN of 01_vanilla_gan.py through
#     ExperimentTracker receipts
#   - Apply to synthetic defect imagery for a manufacturing lab
#
# PREREQUISITES: M5/ex_5/01_vanilla_gan.py (GAN training loop, FID,
#   mode coverage). The MLP GAN there borrows the DCGAN RECIPE; this
#   file implements the DCGAN ARCHITECTURE.
# ESTIMATED TIME: ~35 min
#
# DATASET: MNIST (60K, scaled to [-1, 1]). Trains on CPU: 6 epochs at
#   batch 128 — enough for recognisable digits; say so honestly.
#
# PHASES:
#   1. THEORY  — Why convolutions; the DCGAN guidelines
#   2. BUILD   — ConvGenerator + ConvDiscriminator
#   3. TRAIN   — adversarial loop, tracked with ExperimentTracker
#   4. VISUALISE — sample grid, progression, FID + mode coverage
#   5. APPLY   — synthetic defect imagery for a manufacturing lab
#
# ════════════════════════════════════════════════════════════════════════


In [ ]:
from __future__ import annotations

import copy

import numpy as np
import torch
import torch.nn as nn


device = init_environment()



## PHASE 1 — THEORY: Why Convolutions, and the DCGAN Guidelines


MLP GAN: 784 flat inputs, every spatial relationship relearned.
  DCGAN:   weight sharing + locality match the structure of images.

  Radford et al. (2016) guidelines:
    1. Strided conv (D) / ConvTranspose (G) instead of pooling
    2. BatchNorm in both nets (not G output / D input)
    3. ReLU in G, LeakyReLU(0.2) in D
    4. Adam lr=2e-4, betas=(0.5, 0.999)
    5. No bias into BatchNorm layers (BN cancels it, zero gradient)


In [ ]:
# The MLP GAN of 01_vanilla_gan.py treats a 28x28 digit as a flat
# 784-vector. It works — but it must learn from scratch that pixel
# (i, j) is related to pixel (i, j+1). Every digit position, every
# stroke orientation, is a separate pattern for the MLP.
#
# CONVOLUTION matches the data's structure by construction:
#   - WEIGHT SHARING: one filter detects a curve ANYWHERE in the frame
#   - LOCALITY: a filter reads a 4x4 patch, not all 784 pixels at once
#   - TRANSLATION STRUCTURE: shifting the digit shifts the feature map
#
# THE DCGAN GUIDELINES (Radford, Metz & Chintala, 2016) — empirical
# rules that turned GAN training from alchemy into engineering:
#
#   1. Replace pooling with STRIDED convolutions (D) and
#      FRACTIONALLY-STRIDED convolutions (ConvTranspose, G). The network
#      learns its own up/downsampling instead of a fixed max/avg rule.
#   2. BatchNorm in BOTH networks (except G's output layer and D's input
#      layer) — stabilises the adversarial tug-of-war.
#   3. ReLU in G, LeakyReLU(0.2) in D. Leaky keeps gradients flowing
#      even when D rejects a region confidently.
#   4. Adam with lr=2e-4 and betas=(0.5, 0.999) — the lower first moment
#      damps oscillation in the two-player game.
#   5. No bias on layers feeding BatchNorm: BN subtracts the batch mean,
#      so such a bias is cancelled and receives zero gradient.

print("=" * 70)
print("  PHASE 1 — THEORY: convolutional GANs and the DCGAN guidelines")
print("=" * 70)
print(
)



## PHASE 2 — BUILD: ConvGenerator + ConvDiscriminator


Pure convolutional path (no Linear layers):
        z (B, latent, 1, 1)
        -> ConvTranspose(latent->256, k7, s1, p0)  -> BN -> ReLU  [7x7]
        -> ConvTranspose(256->128,  k4, s2, p1)    -> BN -> ReLU  [14x14]
        -> ConvTranspose(128->1,    k4, s2, p1)    -> Tanh        [28x28]
    No bias on ConvTranspose layers feeding BatchNorm (guideline 5).


Conv(1->64,   k4, s2, p1) -> LeakyReLU        [14x14]  (no BN on input)
        Conv(64->128, k4, s2, p1) -> BN -> LeakyReLU  [7x7]
        Flatten -> Linear(128*7*7 -> 1)


In [ ]:
print("=" * 70)
print("  PHASE 2 — BUILD: the convolutional pair")
print("=" * 70)

DCGAN_EPOCHS = 6
DCGAN_LR = 2e-4
DCGAN_BETAS = (0.5, 0.999)


class ConvGenerator(nn.Module):

    def __init__(self, latent_dim: int = LATENT_DIM):
        super().__init__()
        self.latent_dim = latent_dim
        # TODO: Build the ConvTranspose stack (guideline 1 + 2 + 3 + 5):
        #   1. ConvTranspose2d(latent_dim -> 256, kernel 7, stride 1,
        #      padding 0, NO bias) -> BatchNorm2d -> ReLU     [1x1 -> 7x7]
        #   2. ConvTranspose2d(256 -> 128, kernel 4, stride 2, padding 1,
        #      NO bias) -> BatchNorm2d -> ReLU                [7x7 -> 14x14]
        #   3. ConvTranspose2d(128 -> 1, kernel 4, stride 2, padding 1)
        #      -> Tanh (bias allowed — no BatchNorm after it) [14x14 -> 28x28]
        self.net = nn.Sequential(
            ____,
            nn.BatchNorm2d(256),
            nn.ReLU(),
            ____,
            nn.BatchNorm2d(128),
            nn.ReLU(),
            ____,
            nn.Tanh(),
        )

    def forward(self, z: torch.Tensor) -> torch.Tensor:
        # TODO: reshape the flat (B, latent) noise to (B, latent, 1, 1) so
        #   the convolutional stack can consume it, then run self.net
        return ____


class ConvDiscriminator(nn.Module):

    def __init__(self):
        super().__init__()
        # TODO: Strided-conv feature extractor (NO pooling — guideline 1):
        #   Conv2d(1 -> 64, k4, stride 2, padding 1) + LeakyReLU(0.2)
        #   Conv2d(64 -> 128, k4, stride 2, padding 1, NO bias)
        #     + BatchNorm2d(128) + LeakyReLU(0.2)               [28 -> 14 -> 7]
        self.features = nn.Sequential(
            ____,
            nn.LeakyReLU(0.2),
            ____,
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2),
        )
        # TODO: scalar logit head over the flattened 128*7*7 features
        self.head = ____

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.head(self.features(x).flatten(1))



### Checkpoint 1: architecture shapes and parameter counts


In [ ]:
_g = ConvGenerator().to(device)
_d = ConvDiscriminator().to(device)
_z = torch.randn(4, LATENT_DIM, device=device)
with torch.no_grad():
    _fake = _g(_z)
    _logit = _d(_fake)
assert _fake.shape == (4, 1, 28, 28), f"G output {_fake.shape} != (4, 1, 28, 28)"
assert float(_fake.min()) >= -1.0 and float(_fake.max()) <= 1.0, (
    "Tanh output must stay in [-1, 1]"
)
assert _logit.shape == (4, 1), f"D output {_logit.shape} != (4, 1)"


n_g_conv = sum(p.numel() for p in _g.parameters())
n_d_conv = sum(p.numel() for p in _d.parameters())
n_g_mlp = sum(p.numel() for p in MLPGenerator().parameters())
n_d_mlp = sum(p.numel() for p in MLPDiscriminator().parameters())
print(f"\n  Generator:     {n_g_conv:>10,} params (MLP GAN: {n_g_mlp:,})")
print(f"  Discriminator: {n_d_conv:>10,} params (MLP GAN: {n_d_mlp:,})")
print("  Spatial path: z -> 7x7x256 -> 14x14x128 -> 28x28x1 (all learned)")
print("\n--- Checkpoint 1 passed --- DCGAN architecture verified\n")
del _g, _d, _z, _fake, _logit



## PHASE 3 — TRAIN: the adversarial loop, DCGAN recipe


Adversarial training with the DCGAN recipe, tracked honestly.


In [ ]:
print("=" * 70)
print(f"  PHASE 3 — TRAIN: DCGAN on MNIST ({DCGAN_EPOCHS} epochs, CPU budget)")
print("=" * 70)

X_real, y_real, real_loader = load_mnist(device)
conn, tracker, exp_name, registry = setup_engines()


async def train_dcgan() -> tuple[nn.Module, nn.Module, list[float], list[float], dict]:
    G = ConvGenerator().to(device)
    D = ConvDiscriminator().to(device)
    opt_g = torch.optim.Adam(G.parameters(), lr=DCGAN_LR, betas=DCGAN_BETAS)
    opt_d = torch.optim.Adam(D.parameters(), lr=DCGAN_LR, betas=DCGAN_BETAS)
    bce = nn.BCEWithLogitsLoss()
    g_losses, d_losses = [], []
    snapshots = {0: copy.deepcopy(G.state_dict())}

    async with tracker.track(experiment=exp_name, run_name="dcgan_conv") as run:
        await run.log_params(
            {
                "architecture": "DCGAN_conv",
                "latent_dim": str(LATENT_DIM),
                "lr": str(DCGAN_LR),
                "betas": str(DCGAN_BETAS),
                "epochs": str(DCGAN_EPOCHS),
                "batch_size": "128",
                "loss": "BCEWithLogits (non-saturating G)",
            }
        )
        for epoch in range(DCGAN_EPOCHS):
            eg, ed = [], []
            for (real_batch,) in real_loader:
                bs = real_batch.size(0)

                # Train D: real -> 1, fake -> 0
                z = torch.randn(bs, LATENT_DIM, device=device)
                # TODO: generate fakes WITHOUT building a graph back into G
                # Hint: .detach() the generator output
                fake = ____
                # TODO: D loss — BCE on reals labelled 1 plus BCE on fakes
                #   labelled 0 (BCEWithLogitsLoss, so raw logits)
                loss_d = ____
                opt_d.zero_grad()
                loss_d.backward()
                opt_d.step()

                # Train G (non-saturating): make D call fakes "real"
                z = torch.randn(bs, LATENT_DIM, device=device)
                # TODO: G loss — BCE of D(G(z)) against ONES (the
                #   non-saturating generator objective)
                loss_g = ____
                opt_g.zero_grad()
                loss_g.backward()
                opt_g.step()

                eg.append(loss_g.item())
                ed.append(loss_d.item())

            g_losses.append(float(np.mean(eg)))
            d_losses.append(float(np.mean(ed)))
            await run.log_metrics(
                {"g_loss": g_losses[-1], "d_loss": d_losses[-1]}, step=epoch + 1
            )
            print(
                f"  [DCGAN] epoch {epoch+1}/{DCGAN_EPOCHS}  "
                f"D={d_losses[-1]:.3f}  G={g_losses[-1]:.3f}"
            )
            if (epoch + 1) in {1, 2, 4, DCGAN_EPOCHS}:
                snapshots[epoch + 1] = copy.deepcopy(G.state_dict())
        await run.log_metrics(
            {"final_g_loss": g_losses[-1], "final_d_loss": d_losses[-1]}
        )
    return G, D, g_losses, d_losses, snapshots


G_conv, D_conv, g_losses, d_losses, snapshots  = await train_dcgan()



## DIAGNOSTIC CHECKPOINT — Prescription Pad on both networks


In [ ]:
from kailash_ml.diagnostics import run_diagnostic_checkpoint

_diag_bce = nn.BCEWithLogitsLoss()


def _g_loss(m, batch):
    bs = batch[0].size(0)
    z = torch.randn(bs, LATENT_DIM, device=device)
    return _diag_bce(D_conv(m(z)), torch.ones(bs, 1, device=device))


def _d_loss(m, batch):
    bs = batch[0].size(0)
    with torch.no_grad():
        fake = G_conv(torch.randn(bs, LATENT_DIM, device=device))
    return _diag_bce(m(batch[0]), torch.ones(bs, 1, device=device)) + _diag_bce(
        m(fake), torch.zeros(bs, 1, device=device)
    )


_, g_findings = run_diagnostic_checkpoint(
    G_conv, real_loader, _g_loss, title="DCGAN — Generator",
    n_batches=6, train_losses=g_losses, show=False,
)
print_prescription_pad(g_findings, "DCGAN — Generator")

_, d_findings = run_diagnostic_checkpoint(
    D_conv, real_loader, _d_loss, title="DCGAN — Discriminator",
    n_batches=6, train_losses=d_losses, show=False,
)
print_prescription_pad(d_findings, "DCGAN — Discriminator")



### Checkpoint 2: training converged, game stayed alive


In [ ]:
assert len(g_losses) == DCGAN_EPOCHS
assert g_losses[-1] < g_losses[0], "G loss should decrease as fakes improve"
assert d_losses[-1] > 0.05, (
    f"D loss collapsed to {d_losses[-1]:.4f} — D died, G gets no gradient"
)
print("\n--- Checkpoint 2 passed --- DCGAN trained, adversarial game alive\n")



## PHASE 4 — VISUALISE: samples, progression, FID + mode coverage


In [ ]:
print("=" * 70)
print("  PHASE 4 — VISUALISE: recognisable digits, measured")
print("=" * 70)

# (a) Sample grid from the final generator
G_conv.eval()
with torch.no_grad():
    final_samples = G_conv(torch.randn(64, LATENT_DIM, device=device))
plot_image_grid(
    final_samples,
    title=f"DCGAN samples after {DCGAN_EPOCHS} epochs",
    save_path=str(OUTPUT_DIR / "04_dcgan_samples.png"),
)

# (b) Progression: same fixed z through training snapshots
plot_training_progression(
    G_conv, device, snapshots,
    title="DCGAN training progression (fixed latent vectors)",
    save_path=str(OUTPUT_DIR / "04_dcgan_progression.png"),
)

# (c) Loss dynamics
plot_loss_curves(
    g_losses, d_losses, title="DCGAN training dynamics",
    save_path=str(OUTPUT_DIR / "04_dcgan_losses.png"),
)

# (d) Latent interpolation — evidence of a continuous manifold
plot_latent_interpolation(
    G_conv, device,
    title="DCGAN latent interpolation",
    save_path=str(OUTPUT_DIR / "04_dcgan_interpolation.png"),
)

# (e) Honest quality metrics: FID in LeNet feature space + mode coverage.
# Train the shared feature extractor on MNIST (same protocol as 03).
extractor = train_feature_extractor(X_real, y_real, device)
X_test, y_test = load_mnist_test(device)

with torch.no_grad():
    eval_samples = G_conv(torch.randn(2000, LATENT_DIM, device=device))
    # The trivial baseline every FID number needs: uniform pixel noise in
    # the same [-1, 1] range. "FID 115" means nothing until you know the
    # noise floor — the same discipline as perplexity-vs-vocab in ex_3/07.
    # TODO: 2000 uniform-noise images, shape (2000, 1, 28, 28), in [-1, 1]
    noise_samples = ____
fid = compute_fid(extractor, X_test[:2000], eval_samples)
# TODO: the noise-reference FID — same extractor, same test slice
fid_noise = ____
n_classes_covered, per_class, entropy = mode_coverage(G_conv, extractor, device)

print(f"\n  QUALITY METRICS (measured, this run):")
print(f"    FID (LeNet feature space, 2K samples vs 2K test): {fid:.2f}")
print(f"    FID reference — uniform pixel noise:              {fid_noise:.2f}")
print(f"    Mode coverage: {n_classes_covered}/10 digit classes generated")
print(f"    Class entropy: {entropy:.2f} bits (uniform = 3.32)")
print(
    "    Compare against your 01_vanilla_gan.py run — same extractor "
    "protocol, MLP architecture. The receipts are both in ExperimentTracker."
)

register_generator(registry, "dcgan_generator", G_conv, fid, n_classes_covered,
                   entropy)



### Checkpoint 3: quality floors (conservative, honest)


In [ ]:
import os

assert n_classes_covered >= 6, (
    f"Only {n_classes_covered}/10 digit classes generated — mode collapse"
)
assert entropy > 2.0, f"Class entropy {entropy:.2f} too low — low diversity"
assert np.isfinite(fid) and fid >= 0.0, "FID must be a finite distance"
assert fid < 0.5 * fid_noise, (
    f"FID {fid:.2f} should be far below the noise reference {fid_noise:.2f} — "
    "a trained conv GAN is much closer to real digits than pixel noise is"
)
for artefact in (
    "04_dcgan_samples.png",
    "04_dcgan_progression.png",
    "04_dcgan_losses.png",
    "04_dcgan_interpolation.png",
):
    assert os.path.exists(OUTPUT_DIR / artefact), f"Missing: {artefact}"
print("\n--- Checkpoint 3 passed --- visual proof + honest metrics\n")



## PHASE 5 — APPLY: synthetic defect imagery for a manufacturing lab


SYNTHETIC DATA PROGRAM — QUALIFICATION REPORT (measured, this run):

    Generator:              DCGAN (convolutional, {n_g_conv:,} params)
    Training:               {DCGAN_EPOCHS} epochs on 60K reference images
    FID (LeNet space):      {fid:.2f}  (lower = closer to real distribution)
    FID noise reference:    {fid_noise:.2f}  (uniform pixel noise)
    Mode coverage:          {n_classes_covered}/10 classes
    Class entropy:          {entropy:.2f} bits (uniform = 3.32)

  DECISION RULES THE LAB CAN DEFEND:
    1. Coverage gate: augmentation proceeds only if all classes appear
       ({n_classes_covered}/10 {'meets' if n_classes_covered == 10 else 'does not fully meet'}
       the 10/10 bar this run).
    2. Fidelity gate: FID tracked in the registry; a regression on
       retraining blocks promotion of the new generator.
    3. Real-data gate: final model qualification ALWAYS uses held-out
       real defects. Synthetic data trains; reality validates.

  STAKEHOLDER-READY OUTPUT:
    "The convolutional generator covers {n_classes_covered} of 10 defect
    classes with class entropy {entropy:.2f} bits and FID {fid:.2f} in
    our domain's feature space. Because defects are spatial patterns,
    the convolutional architecture spends its capacity on defect variety
    rather than relearning locality — which is what the MLP generator
    had to do. Synthetic images augment the training set; the
    qualification gate remains real held-out defects."


In [ ]:
# SCENARIO (anonymised, illustrative): a precision-manufacturing lab
# qualifies visual-inspection models. Real defect photos are rare (the
# line is designed NOT to produce defects), so the lab augments training
# data with synthetic defect images.
#
# WHY THE DCGAN ARCHITECTURE MATTERS HERE:
#   Defects are SPATIAL: a scratch is a scratch anywhere on the wafer.
#   An MLP generator wastes capacity relearning "edge" at every pixel
#   offset; a convolutional generator shares that filter everywhere.
#   With the same rare-defect budget, the conv net spends its parameters
#   on VARIETY of defects, not on re-deriving locality.
#
# THE HONEST LIMITS (what we tell the quality manager):
#   - FID is computed in a LeNet feature space trained on the SAME domain
#     (MNIST here). It is comparable across our generators, NOT to FID
#     numbers published with InceptionV3 on natural images.
#   - Mode coverage answers "do we generate all defect types?" — the
#     question that matters for qualification data. A beautiful grid of
#     3 defect types out of 10 is a failed augmentation program.
#   - Synthetic data AUGMENTS real defects; it does not replace the
#     final real-defect validation gate.

print("=" * 70)
print("  PHASE 5 — APPLY: synthetic defect imagery, honestly scoped")
print("=" * 70)
print(
)



### Checkpoint 4: application metrics are the measured ones


In [ ]:
assert 0 < n_classes_covered <= 10
assert fid >= 0.0, "FID is a distance — cannot be negative"
print("--- Checkpoint 4 passed --- synthetic-data application demonstrated\n")

await close_engines(conn)



## REFLECTION


THEORY:
  [x] Convolutions match image structure: weight sharing + locality —
      a stroke detector is learned ONCE and applied everywhere
  [x] DCGAN guidelines (Radford 2016): strided/ConvTranspose instead of
      pooling, BatchNorm in both nets, ReLU(G)/LeakyReLU(D),
      Adam(2e-4, betas=(0.5, 0.999)), no bias into BatchNorm

  BUILD + TRAIN:
  [x] ConvGenerator: z -> 7x7x256 -> 14x14x128 -> 28x28x1, pure
      ConvTranspose ({n_g_conv:,} params vs MLP's {n_g_mlp:,})
  [x] ConvDiscriminator: strided convs to a scalar logit
  [x] {DCGAN_EPOCHS}-epoch adversarial loop, tracked in ExperimentTracker;
      diagnostic Prescription Pad run on BOTH networks

  VISUALISE (the proof):
  [x] Sample grid + fixed-z progression across epochs
  [x] Latent interpolation (continuous manifold evidence)
  [x] Honest metrics: FID {fid:.2f} against a noise reference of
      {fid_noise:.2f} (LeNet space), coverage {n_classes_covered}/10,
      entropy {entropy:.2f} bits

  APPLY:
  [x] Synthetic defect imagery program with three defensible gates:
      coverage, fidelity (registry-tracked FID), and a real-data final
      qualification gate
  [x] Stated limit: synthetic data trains; reality validates

  KEY INSIGHT: "DCGAN" is two things — an ARCHITECTURE (convolutions)
  and a RECIPE (the five guidelines). The MLP GAN in 01 already used the
  recipe; this file showed the architecture. When a GAN underperforms,
  diagnose which half is missing before reaching for a bigger model —
  and let FID + mode coverage, not a pretty grid, deliver the verdict.


In [ ]:
print("\n" + "=" * 70)
print("  WHAT YOU'VE MASTERED")
print("=" * 70)
print(
)

